# Notebook 3 — SpaceX Web Scraping

**IBM Data Science Capstone** · Historical Wikipedia snapshot (9 June 2021).

This notebook extracts historical Falcon 9 / Falcon Heavy launch tables. It does not fabricate launch records. The source page and the scraping date are included for reproducibility.

In [ ]:
%pip -q install requests beautifulsoup4 lxml pandas matplotlib

In [ ]:
import io
import re
from pathlib import Path
from datetime import datetime, timezone

import requests
import pandas as pd
from bs4 import BeautifulSoup
from IPython.display import display

STATIC_URL = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
headers = {"User-Agent": "Mozilla/5.0 (compatible; IBM-SpaceX-Capstone-Education/1.0)"}
response = requests.get(STATIC_URL, headers=headers, timeout=60)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")
print("Page title:", soup.title.get_text(" ", strip=True) if soup.title else "missing")
print("HTTP:", response.status_code)
print("Retrieved UTC:", datetime.now(timezone.utc).isoformat())

In [ ]:
# Identify launch tables by their actual column headers (not fixed table indexes).
def normalize(s):
    return re.sub(r"\s+", " ", str(s)).strip()

launch_tables = []
for table in soup.select("table.wikitable"):
    header_rows = table.find_all("tr", limit=3)
    header_text = " ".join(normalize(th.get_text(" ", strip=True)).lower()
                           for row in header_rows for th in row.find_all("th"))
    if ("flight no" in header_text or "flight number" in header_text) and "payload" in header_text:
        launch_tables.append(table)

print("Matching launch tables:", len(launch_tables))
if not launch_tables:
    raise RuntimeError("No historical launch tables found; inspect the downloaded Wikipedia HTML.")

In [ ]:
# BeautifulSoup locates tables; pandas parses row/column spans into tabular records.
def flatten_columns(columns):
    flat = []
    for col in columns:
        parts = [normalize(c) for c in col] if isinstance(col, tuple) else [normalize(col)]
        parts = [p for p in parts if p and not p.startswith("Unnamed:")]
        flat.append(parts[-1] if parts else "Unknown")
    return flat

frames = []
for idx, table in enumerate(launch_tables, start=1):
    try:
        parsed = pd.read_html(io.StringIO(str(table)), displayed_only=False)[0]
        parsed.columns = flatten_columns(parsed.columns)
        parsed["SourceTable"] = idx
        frames.append(parsed)
    except (ValueError, ImportError) as ex:
        print(f"Skipping table {idx}: {ex}")

if not frames:
    raise RuntimeError("No HTML tables could be parsed.")

raw = pd.concat(frames, ignore_index=True, sort=False)
print("Extracted table shape:", raw.shape)
print("Columns:", raw.columns.tolist())
display(raw.head(10))

In [ ]:
# Check structural variation before making conclusions.
print("Non-null values per column:")
display(raw.notna().sum().sort_values(ascending=False).to_frame("NonNullCount"))

# Keep raw data, including source column names, to avoid silently changing historical records.
out = Path("spacex_web_scraped.csv")
raw.to_csv(out, index=False, encoding="utf-8")
print("Saved:", out.resolve())
print("Rows saved:", len(raw))

In [ ]:
# Optional exploratory overview: assess year coverage from date-like columns.
possible_dates = [c for c in raw.columns if "date" in str(c).lower()]
print("Date-related columns:", possible_dates)
if possible_dates:
    print("Sample date values:")
    display(raw[possible_dates[0]].head(10).to_frame())

# Download file to upload into the data/ folder of your GitHub repository.
try:
    from google.colab import files
    files.download("spacex_web_scraped.csv")
except ImportError:
    print("Outside Google Colab: retrieve spacex_web_scraped.csv from the notebook directory.")

## Interpretation and limitations

- This is a **historical snapshot** of Wikipedia, not a live 2026 dataset.
- This notebook preserves extracted raw table columns. Date formats, merged header names, and supplementary rows may require cleaning in a separate step.
- Report the observed row count and any parsing exclusions; do not presume it equals the 90 observations in the *API-derived* IBM ML dataset.
- GitHub: commit both the executed `.ipynb` and its output CSV.